# L04 — Entities, Resources, Events, and State Variables

**Module**: M02 | **Chapter**: 3 | **Lecture**: L04

## Learning Objectives
By the end of this notebook you will be able to:
1. Classify any component of a system as entity, resource, event, activity, state variable, or performance measure.
2. Apply the entity–resource–event–state framework to two distinct systems (service and manufacturing).
3. Distinguish between exogenous and endogenous variables and justify which cross the model boundary.
4. Produce a complete, self-consistent conceptual model table ready for simulation.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

No SimPy yet — this is a structured thinking lab. Python cells help you organise and check your conceptual model.
---

In [ ]:
import pandas as pd

## 1. The Six Building Blocks

Every discrete-event simulation conceptual model uses exactly six kinds of objects:

| Term | Definition | Key question |
|---|---|---|
| **Entity** | An object that flows through the system and competes for resources | What moves? |
| **Attribute** | A property of an entity that influences its path or timing | What varies between entities? |
| **Resource** | A capacity-limited server that entities must seize | What can be busy or idle? |
| **Event** | An instantaneous change in system state | What happens at a point in time? |
| **Activity** | A duration of time between two events | What takes time? |
| **State variable** | A value that summarises the current system condition | What do you need to know to resume the simulation? |

Performance measures are *derived* from state variables — they are not building blocks themselves.

## 2. System A: University Financial Aid Office

Students arrive at a financial aid office to resolve issues with their accounts.  
There is one triage window (resolves simple issues in ≤5 min) and two counsellor offices (resolve complex issues in 15–45 min).  
After triage, 30% of students are resolved and leave; 70% wait for a counsellor.

**Study question**: *What is the mean student time in office, and what fraction wait > 30 minutes?*

In [ ]:
# Entity table — fill in the '?' entries
entity_table_A = pd.DataFrame([
    {'Entity': 'Student',
     'Key attributes': 'arrival_time, issue_type (simple/complex)',
     'Entry point': 'Office entrance',
     'Exit point': 'After triage (simple) or after counsellor (complex)'},
])

print("Entity table (System A):")
print(entity_table_A.to_string(index=False))
print()
print("Q: Is 'issue_type' determined at arrival, or discovered during triage?")
print("A: ?")

In [ ]:
# Resource table
resource_table_A = pd.DataFrame([
    {'Resource': 'Triage window',
     'Capacity': 1,
     'Service time dist.': 'Exp(mean=3 min)',
     'Discipline': 'FCFS'},
    {'Resource': 'Counsellor office',
     'Capacity': '?',     # fill in
     'Service time dist.': '?',  # triangular? uniform? lognormal?
     'Discipline': 'FCFS'},
])

print("Resource table (System A):")
print(resource_table_A.to_string(index=False))
print()
print("Q: The problem says '15–45 min'. Which distribution family fits? Why?")
print("A: ?")

In [ ]:
# Event list
event_list_A = pd.DataFrame([
    {'Event': 'Student arrival',
     'State changes': 'n_wait_triage += 1; schedule next arrival',
     'Schedules next event': 'Triage-begin (if triage free)'},
    {'Event': 'Triage-begin',
     'State changes': 'triage_busy = True; n_wait_triage -= 1',
     'Schedules next event': 'Triage-end after Exp(3 min)'},
    {'Event': 'Triage-end',
     'State changes': '?',
     'Schedules next event': '? (depends on routing)'},
    {'Event': 'Counsellor-begin',
     'State changes': '?',
     'Schedules next event': '?'},
    {'Event': 'Counsellor-end',
     'State changes': '?',
     'Schedules next event': '?'},
])

print("Event list (System A) — fill in the '?' entries:")
print(event_list_A.to_string(index=False))

In [ ]:
# State variables — minimal set (enough to resume from any point)
state_vars_A = {
    'n_wait_triage':     'number of students waiting for triage',
    'triage_busy':       '0 or 1 — is the triage window occupied?',
    'n_wait_counsellor': '?',
    'n_counsellors_busy':'?',
}

print("Minimum state variable set (System A):")
for var, desc in state_vars_A.items():
    print(f"  {var:25s}: {desc}")

print()
print("Q: Is 'student sojourn time' a state variable or a performance measure?")
print("A: ?")
print()
print("Q: Can you resume a simulation from just these four state variables? What's missing?")
print("A: ?")

In [ ]:
# Quick throughput bound: check stability before simulation
# Arrival rate (assume Poisson)
lam = 10.0   # students/hour

# Triage (all students pass through)
mu_triage = 60.0 / 3.0     # 20 students/hour per window
c_triage  = 1
rho_triage = lam / (c_triage * mu_triage)

# Only 70% reach counsellors
p_complex  = 0.70
mu_counsel = 60.0 / 30.0   # 2 students/hour per counsellor (mean 30 min)
c_counsel  = 2
rho_counsel = (lam * p_complex) / (c_counsel * mu_counsel)

print(f"Arrival rate: {lam:.1f} students/hr")
print(f"Triage utilisation ρ = {rho_triage:.3f}")
print(f"Counsellor utilisation ρ = {rho_counsel:.3f}")
print()
print("Both < 1 → system is stable (no infinite queue growth).")
print("Bottleneck:", 'Triage' if rho_triage > rho_counsel else 'Counsellors')

## 3. System B: CNC Machine Shop

A small machine shop has 3 CNC machines. Jobs arrive in batches (1–4 jobs at a time).  
Each job is assigned to whichever machine has the shortest queue (SPT routing).  
Each machine fails randomly (MTTF = 8 hours) and is repaired by a single technician (repair time ∼ Exp(1 hr)).  
While a machine is under repair, queued jobs wait; in-progress jobs are preempted.

**Study question**: *What is mean job flow time and machine availability under current staffing?*

In [ ]:
# Entity identification — there are TWO entity types here
entity_table_B = pd.DataFrame([
    {'Entity': 'Job',
     'Key attributes': 'arrival_time, batch_id, assigned_machine, priority',
     'Competes for': 'CNC machine'},
    {'Entity': 'Failure event',
     'Key attributes': '?',
     'Competes for': '?'},
])

print("Entity table (System B):")
print(entity_table_B.to_string(index=False))
print()
print("Note: breakdown events are typically modelled as a separate SimPy process,")
print("not as entities. But they do compete for the repair technician resource.")

In [ ]:
# Resource table for System B
resource_table_B = pd.DataFrame([
    {'Resource': 'CNC Machine',
     'Capacity': 3,
     'Service time': '? (given in problem as ∼ Exp)',
     'Failure': 'MTTF=8 hr (Exp)'},
    {'Resource': 'Repair Technician',
     'Capacity': 1,
     'Service time': 'Exp(1 hr) repair time',
     'Failure': 'N/A'},
])

print("Resource table (System B):")
print(resource_table_B.to_string(index=False))
print()
print("Q: With preemption, what attribute does a job need to track?")
print("A: ?")

In [ ]:
# Endogenous vs. exogenous variables
variables_B = pd.DataFrame([
    {'Variable': 'Job arrivals',          'Endogenous/Exogenous': 'Exogenous', 'Reason': 'Driven by customer orders outside the shop'},
    {'Variable': 'Machine failure time',   'Endogenous/Exogenous': '?',         'Reason': '?'},
    {'Variable': 'Job queue length',       'Endogenous/Exogenous': 'Endogenous','Reason': 'Results from arrival/service interaction'},
    {'Variable': 'Repair completion time', 'Endogenous/Exogenous': '?',         'Reason': '?'},
    {'Variable': 'Batch size distribution','Endogenous/Exogenous': 'Exogenous', 'Reason': 'Set by customer order patterns'},
    {'Variable': 'Machine assignment',     'Endogenous/Exogenous': '?',         'Reason': '? (SPT is a policy choice, not an external input)'},
])

print("Endogenous vs. Exogenous (System B) — fill in '?':")
print(variables_B.to_string(index=False))

## 4. Assumptions Document Template

A complete conceptual model document must include an explicit assumptions list.  
Each assumption should state: *what is assumed*, and *what the consequence would be if violated*.

In [ ]:
# Assumptions for System A (financial aid office)
assumptions_A = [
    {
        'Assumption': 'Arrivals follow a Poisson process (constant rate).',
        'Consequence if violated': 'If arrivals cluster (e.g., peak at semester start), '
                                   'actual waits will exceed simulated waits in peak periods.',
    },
    {
        'Assumption': 'Issue type (simple/complex) is determined independently of arrival time.',
        'Consequence if violated': '?',
    },
    {
        'Assumption': 'No student abandons the queue regardless of wait.',
        'Consequence if violated': '?',
    },
    {
        'Assumption': '?',
        'Consequence if violated': '?',
    },
]

print("Assumptions — System A (Financial Aid Office):")
for i, a in enumerate(assumptions_A, 1):
    print(f"\n  {i}. {a['Assumption']}")
    print(f"     If violated: {a['Consequence if violated']}")

## 5. Design Your Own Conceptual Model

Choose one of the following systems and produce a complete conceptual model in the cells below:

- (a) A university library with self-checkout kiosks and help desks
- (b) A ride-share pickup zone with stochastic car and rider arrivals
- (c) A hospital pharmacy dispensing prescriptions

Your model must include:
1. Study question (one sentence)
2. Entity table (with attributes)
3. Resource table (with capacities and service-time distributions)
4. Minimum event list (5+ events)
5. State variable list
6. ≥4 explicit assumptions with consequences

In [ ]:
# Your system choice: (a), (b), or (c)
system_choice = '?'
study_question = '?'

my_entities = pd.DataFrame([
    {'Entity': '?', 'Attributes': '?', 'Entry': '?', 'Exit': '?'},
])

my_resources = pd.DataFrame([
    {'Resource': '?', 'Capacity': '?', 'Service dist.': '?'},
])

my_events = [
    '?',
]

my_state_vars = [
    '?',
]

my_assumptions = [
    '?',
]

print(f"System: {system_choice}")
print(f"Study question: {study_question}")

---
## Try It Yourself

1. **Level of detail**: For the financial aid office, a more detailed model might track *which* counsellor handles each student (if counsellors specialise). What new entities, attributes, resources, and events would this require? Would the additional detail change the answer to the study question?

2. **Routing logic as state**: In the machine shop, SPT (shortest processing time) routing assigns each job to the machine with the shortest current queue. List every state variable that the routing algorithm must read. What would change if you switched to LPT (longest processing time) routing?

3. **From conceptual model to SimPy skeleton**: For System A, write (without running) the SimPy function signatures you would use to implement the model: `def student_process(env, ...)`, `def run_model(...)`. Map each conceptual model element to a SimPy construct (`Resource`, `timeout`, `yield req`, etc.).